# 🛒 Customer Intelligence System
## Phase 4: Machine Learning Models

**Input:** `df_segmented.csv` (from Phase 3)  
**Goal:** Train and evaluate four ML models, generate SHAP explainability plots, and extract actionable business insights.

---
### Model Roadmap

| Target | Model | Type |
|---|---|---|
| `return_rate` (high/low) | Random Forest | Classification |
| `purchase_conversion_rate` | XGBoost | Regression |
| `premium_subscription` | Logistic Regression | Classification |
| `monthly_spend` | Ridge + XGBoost | Regression |

---
### Notebook Sections
1. Setup & Data Loading
2. Feature Preparation & Train/Test Split
3. Model 1 — Random Forest (return_rate classification)
4. Model 2 — XGBoost (purchase_conversion_rate regression)
5. Model 3 — Logistic Regression (premium_subscription)
6. Model 4 — Ridge + XGBoost (monthly_spend regression)
7. SHAP Explainability
8. Model Comparison & Business Insights

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, RocCurveDisplay,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# XGBoost
try:
    import xgboost as xgb
    print(f'XGBoost version: {xgb.__version__}')
except ImportError:
    print('XGBoost not found — installing...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    import xgboost as xgb

# SHAP
try:
    import shap
    print(f'SHAP version: {shap.__version__}')
except ImportError:
    print('SHAP not found — installing...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap

# Plot styling — consistent across all phases
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='muted')
COLORS = sns.color_palette('tab10', 10)

RANDOM_STATE = 42
TEST_SIZE    = 0.20
CV_FOLDS     = 5

print('\n✅ All libraries loaded successfully')

In [ ]:
# Load segmented dataset from Phase 3
FILE_PATH = 'df_segmented.csv'
df = pd.read_csv(FILE_PATH)

print(f'Dataset shape : {df.shape}')
print(f'Columns       : {df.shape[1]}')
print(f'\nCluster distribution:')
print(df['cluster'].value_counts().sort_index())
print(f'\nPersona distribution:')
print(df['persona'].value_counts())

## 2. Feature Preparation & Train/Test Split

We build a shared feature matrix `X` used across all models, with target-specific splits.

**Feature strategy:**
- Drop string/ID columns (`user_id`, `persona`)
- Drop the specific **target column** from X for each model
- Tree models (RF, XGBoost) use raw features — no scaling needed
- Logistic Regression + Ridge use `StandardScaler` inside a Pipeline

In [ ]:
# ── Define all target columns ─────────────────────────────────────────────────
ALL_TARGETS = [
    'return_rate',
    'purchase_conversion_rate',
    'premium_subscription',
    'monthly_spend',
]

# Columns to always drop from features
DROP_COLS = ['user_id', 'persona', 'cluster'] + ALL_TARGETS
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

# Base feature matrix (all numeric, targets excluded)
X_base = df.drop(columns=DROP_COLS).select_dtypes(include=np.number)

print(f'Base feature matrix shape: {X_base.shape}')
print(f'Feature columns          : {X_base.shape[1]}')
print(f'\nAny NaN in features: {X_base.isnull().sum().sum()}')

# Fill any residual NaN (safety net)
X_base = X_base.fillna(X_base.median())

In [ ]:
# ── Target engineering ────────────────────────────────────────────────────────

# Binary target: return_rate → high (1) if above median, else low (0)
rr_median = df['return_rate'].median()
y_return = (df['return_rate'] > rr_median).astype(int)
print(f'return_rate binary split (median={rr_median:.2f}):')
print(f'  Low  (0): {(y_return==0).sum():,}  ({(y_return==0).mean()*100:.1f}%)')
print(f'  High (1): {(y_return==1).sum():,}  ({(y_return==1).mean()*100:.1f}%)')

# Regression targets
y_conversion = df['purchase_conversion_rate'].copy()
y_spend      = df['monthly_spend'].copy()

# Binary classification: premium_subscription (already 0/1)
y_premium = df['premium_subscription'].astype(int).copy()
print(f'\npremium_subscription balance:')
print(f'  Non-premium (0): {(y_premium==0).sum():,}  ({(y_premium==0).mean()*100:.1f}%)')
print(f'  Premium     (1): {(y_premium==1).sum():,}  ({(y_premium==1).mean()*100:.1f}%)')

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def make_splits(X, y, stratify=None):
    """Train/test split with optional stratification."""
    return train_test_split(
        X, y, test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify
    )

def regression_report(y_true, y_pred, label=''):
    """Print regression metrics."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    print(f'  {label}')
    print(f'    MAE   : {mae:.4f}')
    print(f'    RMSE  : {rmse:.4f}')
    print(f'    R²    : {r2:.4f}')
    print(f'    MAPE  : {mape:.2f}%')
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}

def plot_confusion_matrix(cm, labels, title, ax, cmap='Blues'):
    """Annotated confusion matrix."""
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, cbar=False)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

def plot_feature_importance(importances, feature_names, title, ax, top_n=15, color=COLORS[0]):
    """Horizontal bar chart of top N feature importances."""
    imp_df = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(top_n)
    imp_df.sort_values().plot(kind='barh', ax=ax, color=color, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Importance')
    return imp_df

def plot_residuals(y_true, y_pred, title, axes):
    """Actual vs Predicted + Residual distribution."""
    residuals = y_true - y_pred
    axes[0].scatter(y_pred, y_true, alpha=0.2, s=6, color=COLORS[1])
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_title(f'{title} — Actual vs Predicted', fontweight='bold')
    axes[0].legend()
    axes[1].hist(residuals, bins=50, color=COLORS[2], edgecolor='white')
    axes[1].axvline(0, color='red', linestyle='--')
    axes[1].set_title('Residual Distribution', fontweight='bold')
    axes[1].set_xlabel('Residual')

# Store all model results for final comparison
MODEL_RESULTS = {}
print('✅ Helper functions defined')

## 3. Model 1 — Random Forest Classifier
**Target:** `return_rate` (high / low binary)  
**Why RF?** Handles mixed feature types, robust to outliers, no scaling needed, provides native feature importance.

In [ ]:
# ── Data split ────────────────────────────────────────────────────────────────
X_tr1, X_te1, y_tr1, y_te1 = make_splits(X_base, y_return, stratify=y_return)

print(f'Train : {X_tr1.shape}  |  Test : {X_te1.shape}')
print(f'Train class balance: {y_tr1.value_counts().to_dict()}')

In [ ]:
# ── Train Random Forest ───────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_tr1, y_tr1)
y_pred_rf  = rf.predict(X_te1)
y_prob_rf  = rf.predict_proba(X_te1)[:, 1]

# Cross-validation AUC
cv_auc_rf = cross_val_score(rf, X_base, y_return, cv=CV_FOLDS,
                             scoring='roc_auc', n_jobs=1)
print(f'CV ROC-AUC ({CV_FOLDS}-fold): {cv_auc_rf.mean():.4f} ± {cv_auc_rf.std():.4f}')
print(f'Test ROC-AUC             : {roc_auc_score(y_te1, y_prob_rf):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_te1, y_pred_rf, target_names=['Low Return', 'High Return']))

MODEL_RESULTS['RF_return_rate'] = {
    'cv_auc_mean': cv_auc_rf.mean(), 'cv_auc_std': cv_auc_rf.std(),
    'test_auc': roc_auc_score(y_te1, y_prob_rf)
}

In [ ]:
# ── RF Visuals: Confusion Matrix + ROC + Feature Importance ───────────────────
fig = plt.figure(figsize=(20, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
fig.suptitle('Model 1 — Random Forest: return_rate (High/Low)', fontsize=15, fontweight='bold')

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])

# Confusion matrix
cm_rf = confusion_matrix(y_te1, y_pred_rf)
plot_confusion_matrix(cm_rf, ['Low', 'High'], 'Confusion Matrix', ax1, cmap='Blues')

# ROC curve
fpr, tpr, _ = roc_curve(y_te1, y_prob_rf)
auc_val = roc_auc_score(y_te1, y_prob_rf)
ax2.plot(fpr, tpr, lw=2, color=COLORS[0], label=f'ROC AUC = {auc_val:.4f}')
ax2.plot([0,1],[0,1],'k--', lw=1)
ax2.fill_between(fpr, tpr, alpha=0.1, color=COLORS[0])
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve', fontweight='bold')
ax2.legend(loc='lower right')

# Feature importance
plot_feature_importance(rf.feature_importances_, X_base.columns,
                        'Top 15 Feature Importances', ax3, color=COLORS[0])

plt.tight_layout()
plt.show()

## 4. Model 2 — XGBoost Regressor
**Target:** `purchase_conversion_rate`  
**Why XGBoost?** Handles non-linear relationships, built-in regularization, strong on tabular data.

In [ ]:
# ── Data split ────────────────────────────────────────────────────────────────
X_tr2, X_te2, y_tr2, y_te2 = make_splits(X_base, y_conversion)

print(f'Train : {X_tr2.shape}  |  Test : {X_te2.shape}')
print(f'Target stats — Mean: {y_conversion.mean():.3f} | Std: {y_conversion.std():.3f}')

In [ ]:
# ── Train XGBoost Regressor ───────────────────────────────────────────────────
xgb_reg = xgb.XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)
xgb_reg.fit(
    X_tr2, y_tr2,
    eval_set=[(X_te2, y_te2)],
    verbose=False
)
y_pred_xgb = xgb_reg.predict(X_te2)

# Cross-validation R²
cv_r2_xgb = cross_val_score(xgb_reg, X_base, y_conversion,
                              cv=CV_FOLDS, scoring='r2', n_jobs=-1)
print(f'CV R² ({CV_FOLDS}-fold): {cv_r2_xgb.mean():.4f} ± {cv_r2_xgb.std():.4f}')
print(f'\nTest metrics:')
metrics_xgb = regression_report(y_te2, y_pred_xgb, 'XGBoost — purchase_conversion_rate')

MODEL_RESULTS['XGB_conversion'] = {
    'cv_r2_mean': cv_r2_xgb.mean(), 'cv_r2_std': cv_r2_xgb.std(), **metrics_xgb
}

In [ ]:
# ── XGBoost Visuals: Residuals + Feature Importance + Learning Curve ──────────
fig = plt.figure(figsize=(20, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
fig.suptitle('Model 2 — XGBoost: purchase_conversion_rate', fontsize=15, fontweight='bold')

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])

# Actual vs Predicted
ax1.scatter(y_pred_xgb, y_te2, alpha=0.2, s=6, color=COLORS[1])
lims = [min(y_te2.min(), y_pred_xgb.min()), max(y_te2.max(), y_pred_xgb.max())]
ax1.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')
ax1.set_title(f'Actual vs Predicted  (R²={metrics_xgb["R2"]:.3f})', fontweight='bold')
ax1.legend()

# Residuals
residuals = y_te2.values - y_pred_xgb
ax2.hist(residuals, bins=50, color=COLORS[2], edgecolor='white')
ax2.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax2.set_title('Residual Distribution', fontweight='bold')
ax2.set_xlabel('Residual')
ax2.set_ylabel('Count')

# Feature importance
xgb_imp = xgb_reg.feature_importances_
plot_feature_importance(xgb_imp, X_base.columns, 'Top 15 Feature Importances',
                        ax3, color=COLORS[1])

plt.tight_layout()
plt.show()

## 5. Model 3 — Logistic Regression Classifier
**Target:** `premium_subscription` (0 / 1)  
**Why LR?** Interpretable coefficients, probabilistic outputs, fast to train — ideal for binary business decisions.

In [ ]:
# ── Data split ────────────────────────────────────────────────────────────────
X_tr3, X_te3, y_tr3, y_te3 = make_splits(X_base, y_premium, stratify=y_premium)

print(f'Train : {X_tr3.shape}  |  Test : {X_te3.shape}')
print(f'Train class balance: {y_tr3.value_counts().to_dict()}')

In [ ]:
# ── Logistic Regression in a StandardScaler Pipeline ─────────────────────────
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        C=1.0,
        penalty='l2',
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])
lr_pipe.fit(X_tr3, y_tr3)
y_pred_lr = lr_pipe.predict(X_te3)
y_prob_lr = lr_pipe.predict_proba(X_te3)[:, 1]

# Cross-validation
cv_auc_lr = cross_val_score(lr_pipe, X_base, y_premium,
                             cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE),
                             scoring='roc_auc', n_jobs=-1)
print(f'CV ROC-AUC ({CV_FOLDS}-fold): {cv_auc_lr.mean():.4f} ± {cv_auc_lr.std():.4f}')
print(f'Test ROC-AUC             : {roc_auc_score(y_te3, y_prob_lr):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_te3, y_pred_lr, target_names=['Free', 'Premium']))

MODEL_RESULTS['LR_premium'] = {
    'cv_auc_mean': cv_auc_lr.mean(), 'cv_auc_std': cv_auc_lr.std(),
    'test_auc': roc_auc_score(y_te3, y_prob_lr)
}

In [ ]:
# ── LR Visuals: Confusion Matrix + ROC + Coefficient Plot ─────────────────────
fig = plt.figure(figsize=(20, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
fig.suptitle('Model 3 — Logistic Regression: premium_subscription', fontsize=15, fontweight='bold')

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])

# Confusion matrix
cm_lr = confusion_matrix(y_te3, y_pred_lr)
plot_confusion_matrix(cm_lr, ['Free', 'Premium'], 'Confusion Matrix', ax1, cmap='Greens')

# ROC curve
fpr3, tpr3, _ = roc_curve(y_te3, y_prob_lr)
auc3 = roc_auc_score(y_te3, y_prob_lr)
ax2.plot(fpr3, tpr3, lw=2, color=COLORS[2], label=f'ROC AUC = {auc3:.4f}')
ax2.plot([0,1],[0,1],'k--', lw=1)
ax2.fill_between(fpr3, tpr3, alpha=0.1, color=COLORS[2])
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve', fontweight='bold')
ax2.legend(loc='lower right')

# Coefficients (top 15 by abs value)
lr_model = lr_pipe.named_steps['clf']
coefs = pd.Series(lr_model.coef_[0], index=X_base.columns)
top_coefs = coefs.abs().sort_values(ascending=False).head(15).index
coef_vals = coefs[top_coefs].sort_values()
bar_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in coef_vals]
ax3.barh(coef_vals.index, coef_vals.values, color=bar_colors, edgecolor='white')
ax3.axvline(0, color='black', linewidth=0.8)
ax3.set_title('Top 15 Logistic Coefficients', fontweight='bold')
ax3.set_xlabel('Coefficient (log-odds)')

plt.tight_layout()
plt.show()

## 6. Model 4 — Ridge + XGBoost Regression
**Target:** `monthly_spend`  
We train **both** Ridge (linear baseline) and XGBoost (non-linear), then compare their performance.

In [ ]:
# ── Data split ────────────────────────────────────────────────────────────────
X_tr4, X_te4, y_tr4, y_te4 = make_splits(X_base, y_spend)

print(f'Train : {X_tr4.shape}  |  Test : {X_te4.shape}')
print(f'Target — Mean: ${y_spend.mean():,.2f} | Std: ${y_spend.std():,.2f}')

In [ ]:
# ── 6A: Ridge Regression (inside StandardScaler pipeline) ─────────────────────
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0))
])
ridge_pipe.fit(X_tr4, y_tr4)
y_pred_ridge = ridge_pipe.predict(X_te4)

cv_r2_ridge = cross_val_score(ridge_pipe, X_base, y_spend,
                               cv=CV_FOLDS, scoring='r2', n_jobs=-1)
print('── Ridge Regression ──────────────────────────')
print(f'CV R² ({CV_FOLDS}-fold): {cv_r2_ridge.mean():.4f} ± {cv_r2_ridge.std():.4f}')
metrics_ridge = regression_report(y_te4, y_pred_ridge, 'Ridge — monthly_spend')
MODEL_RESULTS['Ridge_spend'] = {'cv_r2_mean': cv_r2_ridge.mean(), **metrics_ridge}

In [ ]:
# ── 6B: XGBoost Regression for monthly_spend ──────────────────────────────────
xgb_spend = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.04,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.05,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)
xgb_spend.fit(X_tr4, y_tr4, eval_set=[(X_te4, y_te4)], verbose=False)
y_pred_xgb_spend = xgb_spend.predict(X_te4)

cv_r2_xgb_spend = cross_val_score(xgb_spend, X_base, y_spend,
                                   cv=CV_FOLDS, scoring='r2', n_jobs=-1)
print('── XGBoost Regression ────────────────────────')
print(f'CV R² ({CV_FOLDS}-fold): {cv_r2_xgb_spend.mean():.4f} ± {cv_r2_xgb_spend.std():.4f}')
metrics_xgb_spend = regression_report(y_te4, y_pred_xgb_spend, 'XGBoost — monthly_spend')
MODEL_RESULTS['XGB_spend'] = {'cv_r2_mean': cv_r2_xgb_spend.mean(), **metrics_xgb_spend}

In [ ]:
# ── Model 4 Visuals: Ridge vs XGBoost comparison ──────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('Model 4 — monthly_spend: Ridge vs XGBoost', fontsize=15, fontweight='bold')

for row, (preds, label, color) in enumerate([
    (y_pred_ridge,     'Ridge',   COLORS[3]),
    (y_pred_xgb_spend, 'XGBoost', COLORS[4])
]):
    r2 = r2_score(y_te4, preds)

    # Actual vs Predicted
    axes[row, 0].scatter(preds, y_te4, alpha=0.15, s=5, color=color)
    lims = [min(y_te4.min(), preds.min()), max(y_te4.max(), preds.max())]
    axes[row, 0].plot(lims, lims, 'r--', linewidth=1.5)
    axes[row, 0].set_title(f'{label} — Actual vs Predicted  (R²={r2:.3f})', fontweight='bold')
    axes[row, 0].set_xlabel('Predicted')
    axes[row, 0].set_ylabel('Actual')

    # Residuals
    residuals = y_te4.values - preds
    axes[row, 1].hist(residuals, bins=60, color=color, edgecolor='white', alpha=0.8)
    axes[row, 1].axvline(0, color='red', linestyle='--', linewidth=1.5)
    axes[row, 1].set_title(f'{label} — Residual Distribution', fontweight='bold')
    axes[row, 1].set_xlabel('Residual')

plt.tight_layout()
plt.show()

# Ridge coefficients
ridge_model = ridge_pipe.named_steps['reg']
ridge_coefs = pd.Series(ridge_model.coef_, index=X_base.columns)
top_ridge   = ridge_coefs.abs().sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Model 4 — Feature Importance Comparison', fontsize=14, fontweight='bold')

coef_plot = ridge_coefs[top_ridge.index].sort_values()
bar_colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in coef_plot]
axes[0].barh(coef_plot.index, coef_plot.values, color=bar_colors, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Ridge — Top 15 Coefficients', fontweight='bold')
axes[0].set_xlabel('Coefficient')

plot_feature_importance(xgb_spend.feature_importances_, X_base.columns,
                        'XGBoost — Top 15 Feature Importances', axes[1], color=COLORS[4])
plt.tight_layout()
plt.show()

## 7. SHAP Explainability

SHAP (SHapley Additive exPlanations) gives us **per-sample, per-feature attributions** — telling us not just *which* features matter, but *how* and *for whom*.

We generate SHAP explanations for:
- **Random Forest** → `return_rate` (TreeExplainer)
- **XGBoost** → `purchase_conversion_rate` + `monthly_spend` (TreeExplainer)
- **Logistic Regression** → `premium_subscription` (LinearExplainer)

In [ ]:
# Sample background for SHAP (faster computation)
SHAP_SAMPLE = min(5_000, len(X_base))
shap_bg = X_base.sample(SHAP_SAMPLE, random_state=RANDOM_STATE)

print(f'SHAP background sample size: {SHAP_SAMPLE:,}')
shap.initjs()

In [ ]:
# ── 7A: SHAP for Random Forest (return_rate) ──────────────────────────────────
print('Computing SHAP values for Random Forest...')
explainer_rf = shap.TreeExplainer(rf, feature_perturbation='interventional',
                                   data=shap_bg)
shap_vals_rf = explainer_rf.shap_values(X_te1.sample(min(2000, len(X_te1)),
                                                       random_state=RANDOM_STATE))

# For binary classifiers, TreeExplainer returns a list [class0, class1]
if isinstance(shap_vals_rf, list):
    shap_rf_pos = shap_vals_rf[1]  # SHAP for class 1 (High Return Rate)
else:
    shap_rf_pos = shap_vals_rf

X_te1_shap = X_te1.sample(min(2000, len(X_te1)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle('SHAP — Random Forest: return_rate (High/Low)', fontsize=14, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(shap_rf_pos, X_te1_shap, show=False, plot_type='bar',
                  max_display=15, color=COLORS[0])
axes[0].set_title('Mean |SHAP| — Feature Importance', fontweight='bold')

plt.sca(axes[1])
shap.summary_plot(shap_rf_pos, X_te1_shap, show=False, max_display=15)
axes[1].set_title('SHAP Beeswarm — Impact on High Return Rate', fontweight='bold')

plt.tight_layout()
plt.show()
print('✅ RF SHAP done')

In [ ]:
# ── 7B: SHAP for XGBoost (purchase_conversion_rate) ──────────────────────────
print('Computing SHAP values for XGBoost (conversion)...')
explainer_xgb = shap.TreeExplainer(xgb_reg)
X_te2_shap = X_te2.sample(min(2000, len(X_te2)), random_state=RANDOM_STATE)
shap_vals_xgb = explainer_xgb.shap_values(X_te2_shap)

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle('SHAP — XGBoost: purchase_conversion_rate', fontsize=14, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(shap_vals_xgb, X_te2_shap, show=False, plot_type='bar',
                  max_display=15, color=COLORS[1])
axes[0].set_title('Mean |SHAP| — Feature Importance', fontweight='bold')

plt.sca(axes[1])
shap.summary_plot(shap_vals_xgb, X_te2_shap, show=False, max_display=15)
axes[1].set_title('SHAP Beeswarm — Impact on Conversion Rate', fontweight='bold')

plt.tight_layout()
plt.show()
print('✅ XGBoost conversion SHAP done')

In [ ]:
# ── 7C: SHAP for XGBoost (monthly_spend) ─────────────────────────────────────
print('Computing SHAP values for XGBoost (spend)...')
explainer_spend = shap.TreeExplainer(xgb_spend)
X_te4_shap = X_te4.sample(min(2000, len(X_te4)), random_state=RANDOM_STATE)
shap_vals_spend = explainer_spend.shap_values(X_te4_shap)

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle('SHAP — XGBoost: monthly_spend', fontsize=14, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(shap_vals_spend, X_te4_shap, show=False, plot_type='bar',
                  max_display=15, color=COLORS[4])
axes[0].set_title('Mean |SHAP| — Feature Importance', fontweight='bold')

plt.sca(axes[1])
shap.summary_plot(shap_vals_spend, X_te4_shap, show=False, max_display=15)
axes[1].set_title('SHAP Beeswarm — Impact on Monthly Spend', fontweight='bold')

plt.tight_layout()
plt.show()
print('✅ XGBoost spend SHAP done')

In [ ]:
# ── 7D: SHAP for Logistic Regression (premium_subscription) ───────────────────
print('Computing SHAP values for Logistic Regression...')
scaler_lr    = lr_pipe.named_steps['scaler']
X_te3_scaled = scaler_lr.transform(X_te3)
bg_scaled    = scaler_lr.transform(shap_bg)

explainer_lr   = shap.LinearExplainer(lr_model, bg_scaled, feature_dependence='independent')
X_te3_shap_sc  = X_te3_scaled[:min(2000, len(X_te3_scaled))]
shap_vals_lr   = explainer_lr.shap_values(X_te3_shap_sc)

X_te3_shap_df = pd.DataFrame(
    scaler_lr.inverse_transform(X_te3_shap_sc),
    columns=X_base.columns
)

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle('SHAP — Logistic Regression: premium_subscription', fontsize=14, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(shap_vals_lr, X_te3_shap_df, show=False, plot_type='bar',
                  max_display=15, color=COLORS[2])
axes[0].set_title('Mean |SHAP| — Feature Importance', fontweight='bold')

plt.sca(axes[1])
shap.summary_plot(shap_vals_lr, X_te3_shap_df, show=False, max_display=15)
axes[1].set_title('SHAP Beeswarm — Impact on Premium Subscription', fontweight='bold')

plt.tight_layout()
plt.show()
print('✅ Logistic Regression SHAP done')

In [ ]:
# ── 7E: SHAP Dependence Plots — top interactions ──────────────────────────────
# Show how the top feature interacts with another to affect predictions

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('SHAP Dependence Plots', fontsize=14, fontweight='bold')

# Top feature from XGBoost spend model
top_feat_spend = pd.Series(xgb_spend.feature_importances_,
                            index=X_base.columns).idxmax()
shap.dependence_plot(top_feat_spend, shap_vals_spend, X_te4_shap,
                     ax=axes[0], show=False)
axes[0].set_title(f'Spend Model — {top_feat_spend} SHAP dependence', fontweight='bold')

# Top feature from XGBoost conversion model
top_feat_conv = pd.Series(xgb_reg.feature_importances_,
                           index=X_base.columns).idxmax()
shap.dependence_plot(top_feat_conv, shap_vals_xgb, X_te2_shap,
                     ax=axes[1], show=False)
axes[1].set_title(f'Conversion Model — {top_feat_conv} SHAP dependence', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Model Comparison & Business Insights

In [ ]:
# ── All classifier ROC curves on one plot ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))

roc_models = [
    ('Random Forest — return_rate',     y_te1, y_prob_rf, COLORS[0]),
    ('Logistic Reg — premium_sub',      y_te3, y_prob_lr, COLORS[2]),
]
for label, y_true, y_prob, color in roc_models:
    fpr_, tpr_, _ = roc_curve(y_true, y_prob)
    auc_ = roc_auc_score(y_true, y_prob)
    ax.plot(fpr_, tpr_, lw=2, color=color, label=f'{label}  (AUC={auc_:.3f})')
    ax.fill_between(fpr_, tpr_, alpha=0.07, color=color)

ax.plot([0,1],[0,1],'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Classifiers', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Regression model R² comparison ───────────────────────────────────────────
reg_compare = pd.DataFrame({
    'Model': ['Ridge (spend)', 'XGBoost (spend)', 'XGBoost (conversion)'],
    'R²': [
        MODEL_RESULTS['Ridge_spend']['R2'],
        MODEL_RESULTS['XGB_spend']['R2'],
        MODEL_RESULTS['XGB_conversion']['R2'],
    ],
    'MAE': [
        MODEL_RESULTS['Ridge_spend']['MAE'],
        MODEL_RESULTS['XGB_spend']['MAE'],
        MODEL_RESULTS['XGB_conversion']['MAE'],
    ],
    'RMSE': [
        MODEL_RESULTS['Ridge_spend']['RMSE'],
        MODEL_RESULTS['XGB_spend']['RMSE'],
        MODEL_RESULTS['XGB_conversion']['RMSE'],
    ],
})

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Regression Model Comparison', fontsize=14, fontweight='bold')

for ax, metric in zip(axes, ['R²', 'MAE', 'RMSE']):
    colors = [COLORS[3], COLORS[4], COLORS[1]]
    bars = ax.bar(reg_compare['Model'], reg_compare[metric],
                  color=colors, edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(reg_compare[metric])*0.01,
                f'{bar.get_height():.3f}',
                ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
display(reg_compare.set_index('Model').round(4))

In [ ]:
# ── Cross-validation score summary ────────────────────────────────────────────
cv_summary = pd.DataFrame([
    {'Model': 'Random Forest',         'Task': 'Classification', 'Target': 'return_rate',
     'CV Metric': 'ROC-AUC',
     'Mean': MODEL_RESULTS['RF_return_rate']['cv_auc_mean'],
     'Std':  MODEL_RESULTS['RF_return_rate']['cv_auc_std'],
     'Test': MODEL_RESULTS['RF_return_rate']['test_auc']},
    {'Model': 'Logistic Regression',   'Task': 'Classification', 'Target': 'premium_subscription',
     'CV Metric': 'ROC-AUC',
     'Mean': MODEL_RESULTS['LR_premium']['cv_auc_mean'],
     'Std':  MODEL_RESULTS['LR_premium']['cv_auc_std'],
     'Test': MODEL_RESULTS['LR_premium']['test_auc']},
    {'Model': 'XGBoost',               'Task': 'Regression',     'Target': 'purchase_conversion_rate',
     'CV Metric': 'R²',
     'Mean': MODEL_RESULTS['XGB_conversion']['cv_r2_mean'],
     'Std':  MODEL_RESULTS['XGB_conversion']['cv_r2_std'],
     'Test': MODEL_RESULTS['XGB_conversion']['R2']},
    {'Model': 'Ridge',                  'Task': 'Regression',     'Target': 'monthly_spend',
     'CV Metric': 'R²',
     'Mean': MODEL_RESULTS['Ridge_spend']['cv_r2_mean'],
     'Std':  MODEL_RESULTS['Ridge_spend']['cv_r2_std'],
     'Test': MODEL_RESULTS['Ridge_spend']['R2']},
    {'Model': 'XGBoost',               'Task': 'Regression',     'Target': 'monthly_spend',
     'CV Metric': 'R²',
     'Mean': MODEL_RESULTS['XGB_spend']['cv_r2_mean'],
     'Std':  MODEL_RESULTS['XGB_spend']['cv_r2_std'],
     'Test': MODEL_RESULTS['XGB_spend']['R2']},
])

print('=== CROSS-VALIDATION SUMMARY ===')
display(cv_summary.set_index(['Model', 'Target']).style
        .background_gradient(cmap='RdYlGn', subset=['Mean', 'Test'])
        .format({'Mean': '{:.4f}', 'Std': '{:.4f}', 'Test': '{:.4f}'}))

In [ ]:
# ── Predicted spend by persona (business insight) ─────────────────────────────
df_pred = df.copy()
df_pred['predicted_spend']      = xgb_spend.predict(X_base)
df_pred['predicted_conversion'] = xgb_reg.predict(X_base)
df_pred['predicted_premium_prob'] = lr_pipe.predict_proba(X_base)[:, 1]
df_pred['predicted_high_return']  = rf.predict_proba(X_base)[:, 1]

persona_preds = df_pred.groupby('persona')[[
    'predicted_spend', 'predicted_conversion',
    'predicted_premium_prob', 'predicted_high_return'
]].mean().round(3)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Model Predictions by Customer Persona', fontsize=15, fontweight='bold')

metrics_info = [
    ('predicted_spend',        'Avg Predicted Monthly Spend ($)', COLORS[0]),
    ('predicted_conversion',   'Avg Predicted Conversion Rate',   COLORS[1]),
    ('predicted_premium_prob', 'Avg P(Premium Subscription)',     COLORS[2]),
    ('predicted_high_return',  'Avg P(High Return Rate)',         COLORS[3]),
]

for ax, (col, title, color) in zip(axes.flat, metrics_info):
    data = persona_preds[col].sort_values(ascending=True)
    bars = ax.barh(data.index, data.values, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Value')
    for bar in bars:
        ax.text(bar.get_width() + max(data)*0.005, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\n📊 Persona Predictions Table:')
display(persona_preds.style.background_gradient(cmap='RdYlGn'))

In [ ]:
# ── Save model predictions to disk ────────────────────────────────────────────
OUTPUT_PREDS = 'df_predictions.csv'
df_pred.to_csv(OUTPUT_PREDS, index=False)
print(f'✅ Predictions saved: {OUTPUT_PREDS}  {df_pred.shape}')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('=' * 70)
print('         🤖 ML MODELS — FINAL SUMMARY')
print('=' * 70)

print('\n📊 MODEL PERFORMANCE')
print(f'  ┌ Random Forest  (return_rate)          AUC = {MODEL_RESULTS["RF_return_rate"]["test_auc"]:.4f}')
print(f'  ├ Logistic Reg  (premium_subscription)  AUC = {MODEL_RESULTS["LR_premium"]["test_auc"]:.4f}')
print(f'  ├ XGBoost       (conversion_rate)        R²  = {MODEL_RESULTS["XGB_conversion"]["R2"]:.4f}')
print(f'  ├ Ridge         (monthly_spend)          R²  = {MODEL_RESULTS["Ridge_spend"]["R2"]:.4f}')
print(f'  └ XGBoost       (monthly_spend)          R²  = {MODEL_RESULTS["XGB_spend"]["R2"]:.4f}')

print('\n🔍 KEY SHAP INSIGHTS')
top_rf   = pd.Series(rf.feature_importances_, index=X_base.columns).idxmax()
top_xgb  = pd.Series(xgb_reg.feature_importances_, index=X_base.columns).idxmax()
top_sp   = pd.Series(xgb_spend.feature_importances_, index=X_base.columns).idxmax()
top_lr   = pd.Series(np.abs(lr_model.coef_[0]), index=X_base.columns).idxmax()
print(f'  • return_rate top driver         : {top_rf}')
print(f'  • conversion_rate top driver     : {top_xgb}')
print(f'  • monthly_spend top driver       : {top_sp}')
print(f'  • premium_subscription top driver: {top_lr}')

print('\n💡 BUSINESS RECOMMENDATIONS')
print('  1. Target Loyal Whales with premium upsell — highest P(premium) predicted')
print('  2. Reduce return friction for High-Risk segment via better product descriptions')
print('  3. Engagement nudges (push/email) most effective for Window Shoppers')
print('  4. Conversion campaigns should focus on features with high SHAP |value|')
print('  5. Use predicted_spend to prioritize loyalty rewards for mid-tier segments')

print('\n📦 OUTPUT FILES')
print('  • df_predictions.csv — all model predictions + persona labels')

print('\n' + '=' * 70)
print('✅ Phase 4 Complete — End-to-End Customer Intelligence System Done!')
print('=' * 70)

---
## ✅ Project Complete

| Phase | Notebook | Status |
|-------|----------|--------|
| Phase 1 | `01_EDA.ipynb` | ✅ Distributions, correlations, outliers, insights |
| Phase 2 | `02_Preprocessing.ipynb` | ✅ Encoding, scaling, feature engineering |
| Phase 3 | `03_Segmentation.ipynb` | ✅ KMeans, PCA, personas, heatmaps |
| Phase 4 | `04_ML_Models.ipynb` | ✅ RF, XGBoost, LR, Ridge, SHAP, persona predictions |